In [1]:
## 0) Install & Imports

In [2]:
import json
from pathlib import Path
from typing import Dict, List, Any

import pandas as pd
from tqdm import tqdm

from gliner2 import GLiNER2

print("Imports OK ✅")

C:\Users\super\Documents\UniPd\ATA\GutBrainIE\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Imports OK ✅


## 1) Configuration

In [3]:
# -----------------------------
# Paths (EDIT ME)
# -----------------------------
DATA_DIR = Path(r"C:/Users/super/Documents/UniPd/ATA/GutBrainIE/data/GutBrainIE_Full_Collection_2025/Annotations")

# Example: dev gold (has 'entities')
DEV_JSON = DATA_DIR / "Dev/json_format/dev.json"

# Output dir
OUT_DIR = Path(r"C:/Users/super/Documents/UniPd/ATA/GutBrainIE/src/predictions")

# Model
GLINER2_MODEL_NAME = "fastino/gliner2-base-v1"

# Inference
THRESHOLD = 0.30
INCLUDE_CONFIDENCE = True

print("DEV_JSON:", DEV_JSON)
print("OUT_DIR:", OUT_DIR.resolve())

DEV_JSON: C:\Users\super\Documents\UniPd\ATA\GutBrainIE\data\GutBrainIE_Full_Collection_2025\Annotations\Dev\json_format\dev.json
OUT_DIR: C:\Users\super\Documents\UniPd\ATA\GutBrainIE\src\predictions


## 2) Entity labels (GutBrainIE 6.1.1)
We keep the official 13 categories. We map `disease, disorder or finding` → `DDF` to match the dataset labels.

In [4]:
ENTITY_LABELS: List[str] = [
    "anatomical location",
    "animal",
    "bacteria",
    "biomedical technique",
    "chemical",
    "disease, disorder or finding",  # will normalize to DDF
    "dietary supplement",
    "drug",
    "food",
    "gene",
    "human",
    "microbiome",
    "statistical technique",
]

LABEL_MAPPING = {"disease, disorder or finding": "DDF"}

def normalize_label(label: str) -> str:
    return LABEL_MAPPING.get(label, label)

LEGAL_ENTITY_LABELS = set([
    "anatomical location", "animal", "bacteria", "biomedical technique",
    "chemical", "DDF", "dietary supplement", "drug", "food", "gene",
    "human", "microbiome", "statistical technique"
])

print(f"Loaded {len(ENTITY_LABELS)} labels ✅")


Loaded 13 labels ✅


## 3) Load data

In [5]:
def load_ner_data(file_paths: List[Path]) -> Dict[str, Any]:
    all_data: Dict[str, Any] = {}
    for fp in file_paths:
        fp = Path(fp)
        if not fp.exists():
            print(f"⚠️ Missing: {fp}")
            continue
        with fp.open("r", encoding="utf-8") as f:
            data = json.load(f)
        all_data.update(data)
        print(f"Loaded {len(data)} docs from {fp.name}")
    return all_data

dev_data = load_ner_data([DEV_JSON])

Loaded 40 docs from dev.json


### Peek one document

In [6]:
example_pmid = next(iter(dev_data.keys()))
ex = dev_data[example_pmid]

print("PMID:", example_pmid)
print("Title:", ex["metadata"]["title"][:120], "...")
print("Abstract:", ex["metadata"]["abstract"][:180], "...")
print("Gold entities:", len(ex.get("entities", [])))

PMID: 36532064
Title: Hypothesis of a potential BrainBiota and its relation to CNS autoimmune inflammation. ...
Abstract: Infectious agents have been long considered to play a role in the pathogenesis of neurological diseases as part of the interaction between genetic susceptibility and the environmen ...
Gold entities: 19


## 4) Load GLiNER2 model

In [7]:
extractor = GLiNER2.from_pretrained(GLINER2_MODEL_NAME)
print("Model loaded ✅", GLINER2_MODEL_NAME)

You are using a model of type extractor to instantiate a model of type . This is not supported for all configurations of models and can yield errors.


🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first
Model loaded ✅ fastino/gliner2-base-v1


## 5) Inference helpers
**Offsets note:** GLiNER2 returns spans with `end` typically **exclusive**. GutBrainIE JSON offsets are usually **inclusive** → we convert `end_inclusive = end_exclusive - 1`.

In [8]:
def gliner2_extract_spans(
    extractor: GLiNER2,
    text: str,
    labels: List[str],
    threshold: float,
    location: str,
    include_confidence: bool = True,
) -> List[Dict[str, Any]]:
    if not text:
        return []

    result = extractor.extract_entities(
        text,
        labels,
        threshold=threshold,
        include_spans=True,
        include_confidence=include_confidence,
    )

    formatted: List[Dict[str, Any]] = []
    for raw_label, items in result.get("entities", {}).items():
        norm_label = normalize_label(raw_label)
        if norm_label not in LEGAL_ENTITY_LABELS:
            continue

        for it in items:
            start = int(it["start"])
            end_exclusive = int(it["end"])
            end_inclusive = end_exclusive - 1

            formatted.append({
                "start_idx": start,
                "end_idx": end_inclusive,
                "location": location,
                "text_span": it["text"],
                "label": norm_label,
                "score": float(it.get("confidence", 1.0)),
            })

    return formatted


## 6) Post-processing
- Dedupe by (start,end,location,label)
- Remove overlaps (keep longest span; ties: higher score)
- Merge adjacent spans with same label (optional but often helpful)

In [9]:
def dedupe_entities(ents: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    seen = set()
    out = []
    for e in ents:
        k = (e["start_idx"], e["end_idx"], e["location"], e["label"])
        if k in seen:
            continue
        seen.add(k)
        out.append(e)
    return out

def prune_overlaps_keep_longest(ents: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    by_loc: Dict[str, List[Dict[str, Any]]] = {}
    for e in ents:
        by_loc.setdefault(e["location"], []).append(e)

    kept_all: List[Dict[str, Any]] = []
    for loc, group in by_loc.items():
        group = sorted(group, key=lambda x: (x["start_idx"], -(x["end_idx"]-x["start_idx"]), -x.get("score", 1.0)))

        clusters = []
        cur = []
        cur_end = None

        for e in group:
            s, t = e["start_idx"], e["end_idx"]
            if not cur:
                cur = [e]
                cur_end = t
            else:
                if s <= cur_end:  # overlap (inclusive end)
                    cur.append(e)
                    cur_end = max(cur_end, t)
                else:
                    clusters.append(cur)
                    cur = [e]
                    cur_end = t
        if cur:
            clusters.append(cur)

        for cl in clusters:
            best = max(cl, key=lambda x: ((x["end_idx"] - x["start_idx"]), x.get("score", 1.0)))
            kept_all.append(best)

    return kept_all

def merge_adjacent_same_label(ents: List[Dict[str, Any]], title: str, abstract: str) -> List[Dict[str, Any]]:
    by_loc: Dict[str, List[Dict[str, Any]]] = {}
    for e in ents:
        by_loc.setdefault(e["location"], []).append(e)

    merged_all: List[Dict[str, Any]] = []
    for loc, group in by_loc.items():
        text = title if loc == "title" else abstract
        group = sorted(group, key=lambda x: (x["label"], x["start_idx"], x["end_idx"]))

        i = 0
        while i < len(group):
            cur = dict(group[i])
            i += 1
            while i < len(group) and group[i]["label"] == cur["label"]:
                nxt = group[i]
                if nxt["start_idx"] == cur["end_idx"] + 1:
                    cur["end_idx"] = nxt["end_idx"]
                    i += 1
                    continue
                if nxt["start_idx"] == cur["end_idx"] + 2 and text and text[cur["end_idx"] + 1:cur["end_idx"] + 2] == " ":
                    cur["end_idx"] = nxt["end_idx"]
                    i += 1
                    continue
                break

            if text and 0 <= cur["start_idx"] <= cur["end_idx"] < len(text):
                cur["text_span"] = text[cur["start_idx"]:cur["end_idx"] + 1]
            merged_all.append(cur)

    return merged_all

def postprocess_entities(ents: List[Dict[str, Any]], title: str, abstract: str) -> List[Dict[str, Any]]:
    ents = dedupe_entities(ents)
    ents = prune_overlaps_keep_longest(ents)
    ents = merge_adjacent_same_label(ents, title=title, abstract=abstract)
    ents = dedupe_entities(ents)
    return ents

## 7) Run inference on dev set

In [10]:
def predict_dataset_gliner2(
    extractor: GLiNER2,
    dataset: Dict[str, Any],
    labels: List[str],
    threshold: float,
) -> Dict[str, Any]:
    preds: Dict[str, Any] = {}

    for pmid, article in tqdm(dataset.items(), desc="GLiNER2 inference"):
        title = article["metadata"].get("title", "") or ""
        abstract = article["metadata"].get("abstract", "") or ""

        ents = []
        ents += gliner2_extract_spans(extractor, title, labels, threshold, location="title", include_confidence=INCLUDE_CONFIDENCE)
        ents += gliner2_extract_spans(extractor, abstract, labels, threshold, location="abstract", include_confidence=INCLUDE_CONFIDENCE)

        ents = postprocess_entities(ents, title=title, abstract=abstract)

        preds[pmid] = {"entities": [
            {k: e[k] for k in ["start_idx", "end_idx", "location", "text_span", "label"]}
            for e in ents
        ]}

    return preds

preds_dev = predict_dataset_gliner2(extractor, dev_data, ENTITY_LABELS, THRESHOLD)
print("Done ✅  predicted entities:", sum(len(v["entities"]) for v in preds_dev.values()))


GLiNER2 inference: 100%|██████████| 40/40 [00:22<00:00,  1.77it/s]

Done ✅  predicted entities: 1039


## 8) Save predictions (evaluation-ready)

In [11]:
out_path = OUT_DIR / f"gliner_v2_NER.json"
with out_path.open("w", encoding="utf-8") as f:
    json.dump(preds_dev, f, ensure_ascii=False, indent=2)

print("Saved ✅", out_path.resolve())

Saved ✅ C:\Users\super\Documents\UniPd\ATA\GutBrainIE\src\predictions\gliner_v2_NER.json


## 9) quick exact-match evaluation on dev
Exact match on (start,end,location,text_span,label).

In [12]:
from collections import Counter

def evaluate_ner_exact(predictions: Dict[str, Any], ground_truth: Dict[str, Any]) -> Dict[str, float]:
    gt_by_pmid = {}
    gt_label_counts = Counter()

    for pmid, article in ground_truth.items():
        gt_by_pmid[pmid] = set()
        for e in article.get("entities", []):
            tup = (int(e["start_idx"]), int(e["end_idx"]), str(e["location"]), str(e["text_span"]), str(e["label"]))
            gt_by_pmid[pmid].add(tup)
            gt_label_counts[e["label"]] += 1

    pred_label_counts = Counter()
    tp_label_counts = Counter()

    for pmid, doc in predictions.items():
        for e in doc.get("entities", []):
            label = e["label"]
            if label not in LEGAL_ENTITY_LABELS:
                continue
            pred_label_counts[label] += 1
            tup = (int(e["start_idx"]), int(e["end_idx"]), str(e["location"]), str(e["text_span"]), str(e["label"]))
            if tup in gt_by_pmid.get(pmid, set()):
                tp_label_counts[label] += 1

    tp = sum(tp_label_counts.values())
    pred_total = sum(pred_label_counts.values())
    gt_total = sum(gt_label_counts.values())

    micro_p = tp / (pred_total + 1e-10)
    micro_r = tp / (gt_total + 1e-10)
    micro_f1 = 2 * micro_p * micro_r / (micro_p + micro_r + 1e-10)

    labels = list(gt_label_counts.keys())
    macro_p = macro_r = macro_f1 = 0.0
    for lab in labels:
        p = tp_label_counts[lab] / (pred_label_counts[lab] + 1e-10)
        r = tp_label_counts[lab] / (gt_label_counts[lab] + 1e-10)
        f1 = 2 * p * r / (p + r + 1e-10)
        macro_p += p
        macro_r += r
        macro_f1 += f1

    n = max(1, len(labels))
    macro_p /= n
    macro_r /= n
    macro_f1 /= n

    return {
        "macro_precision": macro_p,
        "macro_recall": macro_r,
        "macro_f1": macro_f1,
        "micro_precision": micro_p,
        "micro_recall": micro_r,
        "micro_f1": micro_f1,
        "tp": tp,
        "pred_total": pred_total,
        "gt_total": gt_total,
    }

metrics = evaluate_ner_exact(preds_dev, dev_data)
metrics

{'macro_precision': 0.37777350816050304,
 'macro_recall': 0.4250602645066862,
 'macro_f1': 0.3677356507856106,
 'micro_precision': 0.43599615014432763,
 'micro_recall': 0.40555058191580967,
 'micro_f1': 0.4202226344583752,
 'tp': 453,
 'pred_total': 1039,
 'gt_total': 1117}